# Fall & Call: Emergency Keyword Spotting (KWS) v2.0
This notebook trains a CNN model to recognize **"HELP"**, **"EMERGENCY"**, **"CANCEL"** and **"BACKGROUND NOISE"**.

It follows the standard TinyML workflow used in the gender classification project, ensuring compatibility with the Nano 33 BLE Sense audio capture.

### **Model Update**: This version uses `SeparableConv2D` for improved parameter efficiency on microcontrollers.

In [ ]:
import numpy as np
import pandas as pd
import librosa
import librosa.display
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, SeparableConv2D, MaxPooling2D, GlobalAveragePooling2D, Conv2D, Flatten
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
import seaborn as sns
import os
import shutil
import json

# Set MFCC parameters (Matching Lab 4 / Nano 33 BLE Sense specs)
N_MFCC = 13 
N_MELS = 26
SAMPLING_RATE = 16000 # 16 kHz
FRAME_SIZE = 512 # 32 ms
HOP_LENGTH = 256 # 16 ms
MIN_FREQ = 50
MAX_FREQ = SAMPLING_RATE / 2

# 1 second of audio at 16kHz = 16000 samples
# With center=False, n_frames = (16000 - 512) // 256 + 1 = 61
N_FRAMES = 61

CLASSES = ['help', 'emergency', 'cancel', 'background']

**1. Preparing the Data**

In [ ]:
def normalize_audio(audio):
    # Perform peak normalization to range [-1, 1]
    audio = audio.astype(np.float32)
    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio /= max_val
    return audio

def extract_mfcc(data):
    # Compute MFCC with parameters matching the Arduino inference implementation
    mfcc = librosa.feature.mfcc(
        y=data, 
        sr=SAMPLING_RATE,   
        n_mfcc=N_MFCC,  
        n_mels=N_MELS,  
        n_fft=FRAME_SIZE,  
        hop_length=HOP_LENGTH,  
        fmin=MIN_FREQ,  
        fmax=MAX_FREQ,  
        window='hann',
        center=False,  
        dct_type=2,  
        norm='ortho',  
        power=2
    )
    return mfcc

def load_and_process_data(filename, label):
    print(f"Processing {filename}...")
    # Load raw text data (ignoring non-numeric lines like headers)
    try:
        df = pd.read_csv(filename, header=None, on_bad_lines='skip')
        df = df.apply(pd.to_numeric, errors='coerce')
        raw_samples = df.dropna().values.flatten()
        
        num_samples_per_clip = 16000
        num_clips = len(raw_samples) // num_samples_per_clip
        
        mfccs = []
        labels = []
        
        for i in range(num_clips):
            clip = raw_samples[i*num_samples_per_clip : (i+1)*num_samples_per_clip]
            normalized_clip = normalize_audio(clip)
            mfcc = extract_mfcc(normalized_clip)
            
            # Ensure the MFCC has the expected number of frames
            if mfcc.shape[1] == N_FRAMES:
                mfccs.append(mfcc)
                labels.append(label)
        
        return np.array(mfccs), np.array(labels)
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return np.array([]), np.array([])

# Load all classes
all_mfccs = []
all_labels = []

for cls in CLASSES:
    X, y = load_and_process_data(f"{cls}.txt", cls)
    if len(X) > 0:
        all_mfccs.append(X)
        all_labels.append(y)

if all_mfccs:
    mfccs = np.vstack(all_mfccs)
    labels = np.concatenate(all_labels)
    
    print('\nData Extraction Summary:')
    print('MFCCs shape:', mfccs.shape)
    print('Labels shape:', labels.shape)
    
    # Save the processed data
    np.save('mfccs.npy', mfccs)
    np.save('labels.npy', labels)
else:
    print("No data found. Please ensure .txt files are present in the same directory.")

**2. Processing the Data**

In [ ]:
# Load data
mfccs = np.load('mfccs.npy')
labels = np.load('labels.npy')

# Balance classes if necessary (similar to gender lab)
df_data = pd.DataFrame({'mfccs': list(mfccs), 'labels': labels})
print("\nClass distribution before balancing:")
print(df_data['labels'].value_counts())

min_samples = df_data['labels'].value_counts().min()
df_balanced = df_data.groupby('labels').apply(lambda x: x.sample(min_samples)).reset_index(drop=True)

print("\nClass distribution after balancing:")
print(df_balanced['labels'].value_counts())

# Split the dataset (70% Train, 15% Val, 15% Test)
df_train, df_temp = train_test_split(df_balanced, test_size=0.3, stratify=df_balanced['labels'], random_state=42)
df_val, df_test = train_test_split(df_temp, test_size=0.5, stratify=df_temp['labels'], random_state=42)

# Convert back to arrays and Reshape for CNN
train_mfccs = np.array(df_train['mfccs'].tolist()).reshape((-1, N_MFCC, N_FRAMES, 1))
val_mfccs = np.array(df_val['mfccs'].tolist()).reshape((-1, N_MFCC, N_FRAMES, 1))
test_mfccs = np.array(df_test['mfccs'].tolist()).reshape((-1, N_MFCC, N_FRAMES, 1))

# Label Encoding
label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(df_train['labels'])
val_labels = label_encoder.transform(df_val['labels'])
test_labels = label_encoder.transform(df_test['labels'])

print(f"\nMapping: {dict(enumerate(label_encoder.classes_))}")
print(f"Train shape: {train_mfccs.shape}")

In [ ]:
# Generate Mel Filter Bank for C++ reference
mel_filter_bank = librosa.filters.mel(sr=SAMPLING_RATE, n_mels=N_MELS, n_fft=FRAME_SIZE, fmin=MIN_FREQ, fmax=MAX_FREQ)

with open("mel_filter_bank.h", "w") as f:
    f.write("#ifndef MEL_FILTER_BANK_H\n#define MEL_FILTER_BANK_H\n\n")
    f.write(f"#define NUM_MEL_FILTERS {N_MELS}\n")
    f.write(f"#define N_FFT {FRAME_SIZE}\n\n")
    f.write("const float mel_filter_bank[NUM_MEL_FILTERS][N_FFT / 2 + 1] = {\n")
    for row in mel_filter_bank:
        f.write("    {" + ", ".join(map(str, row)) + "},\n")
    f.write("};\n\n#endif")

**3. Training the Model**

In [ ]:
# Lightweight CNN Optimized for TinyML using Separable Convolutions
model = Sequential([
    Input(shape=(N_MFCC, N_FRAMES, 1)),

    SeparableConv2D(16, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)), 

    SeparableConv2D(32, (3,3), activation='relu', padding='same'),
    Dropout(0.2),
    MaxPooling2D((2,2)),  

    SeparableConv2D(64, (3,3), activation='relu', padding='same'), 
    MaxPooling2D((2,2)), 

    GlobalAveragePooling2D(),
    Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),

    Dense(len(CLASSES), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

history = model.fit(
    train_mfccs, train_labels,
    validation_data=(val_mfccs, val_labels),
    epochs=100,
    batch_size=8,
    callbacks=[early_stopping, lr_scheduler]
)

**4. Evaluating the Model**

In [ ]:
# Plot Training History
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()
plt.show()

# Confusion Matrix
y_pred = np.argmax(model.predict(test_mfccs), axis=1)
cm = confusion_matrix(test_labels, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Purples')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

**5. Conversion to TFLite (Quantized)**

In [ ]:
def representative_data_gen():
    for mfcc in test_mfccs.astype(np.float32):
        yield [np.expand_dims(mfcc, axis=0)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()

with open("emergency_model_quant.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Quantized Model Size: {len(tflite_model)} bytes")

**6. Conversion to C Header File**

In [ ]:
def hex_to_c_array(hex_data, var_name):
    c_str = f'#ifndef {var_name.upper()}_H\n#define {var_name.upper()}_H\n\n'
    c_str += f'unsigned int {var_name}_len = {len(hex_data)};\n'
    c_str += f'unsigned char {var_name}[] = {{'
    for i, val in enumerate(hex_data):
        if i % 12 == 0: c_str += '\n  '
        c_str += f'{val:#04x}'
        if i + 1 < len(hex_data): c_str += ', '
    c_str += '\n};\n\n#endif'
    return c_str

with open("emergency_model.h", "w") as f:
    f.write(hex_to_c_array(tflite_model, "emergency_model"))

print("Exported to emergency_model.h")